<a href="https://colab.research.google.com/github/anujbl29-portfolio/tpm-risk-portfolio/blob/main/KYC%20onboarding%20Case%20Study%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Streamlining User Onboarding Verification Journey <br><br>


1. **Executive Summary** <br>

The onboarding KYC journey was a major conversion bottleneck in a high-volume financial product where users could not deposit funds until document verification was completed. Weekly analysis showed that out of 55,000 uploaded documents, approximately 18,000 were being rejected, with the largest drivers being name mismatches between Aadhaar and PAN, low image confidence, and missing/invalid front-back images. At the same time, around 2,000 approved documents (~5%) were identified as forged or edited, exposing the business to fraud and compliance risk.

I led the diagnostic and requirement-gathering phase using a mixed-method approach that combined rejection trend analysis, user-level retry behaviour, funnel drop-off analysis, and fraud-pattern review. Based on these findings, I partnered with Product, Engineering, and the external verification vendor to redesign the KYC workflow through fuzzy name-matching logic, OCR training improvements, live-capture controls, and authenticated document APIs with OTP verification.

This intervention increased overall approval rate from 68% to 81%, reduced verification latency by 60% to under 10 seconds, enabled 10.5K additional activated users, and contributed approximately $150K in additional weekly deposits. <br>


2. **Business Context**<br>

This product operated in a regulated financial environment where users were required to complete KYC before being allowed to deposit money.

For international context:

- Aadhaar is India’s government-issued biometric identity document.

- PAN (Permanent Account Number) is India’s government-issued tax identity document used for financial transactions.

Both documents are commonly used for identity verification, mismatches between the two created both false rejections and fraud-detection challenges.

The KYC system therefore had to balance four competing objectives:

- reduce friction for genuine users

- prevent forged/edited documents from passing

- maintain acceptable API latency

- protect personally identifiable information (PII)

3. **Problem Statement**

The existing KYC workflow was failing in two directions at once:

1. Good users were being rejected too often

- especially because of name-format inconsistencies between Aadhaar and PAN

- poor image quality

- incomplete document uploads

2. Some bad documents were still being approved

- forged or edited images were slipping through the workflow

- this created fraud and compliance risk

This meant the system was simultaneously:

- hurting conversion

- increasing support workload

- creating revenue leakage

- allowing false negatives on document fraud

**Diagnostic Analysis**

Weekly Snapshot

| Metric                         | Volume      |
| ------------------------------ | ----------- |
| Documents Uploaded             | 55,000      |
| Approved                       | 37,000      |
| Rejected                       | 18,000      |
| Forged/Edited (False Negative) | ~2,000 (5%) |

</br>



| Rejection Reason               | Volume |
| ------------------------------ | ------ |
| Name mismatch (Aadhaar vs PAN) | 8,000  |
| Low image confidence           | 4,000  |
| Missing front/back image       | 3,000  |
| Other validation issues        | 3,000  |


</br>

What I Analyzed

I used both quantitative and qualitative analysis:

**Quantitative**

- total rejection volume

- rejection reasons

- user-level document upload attempts

- funnel drop-off after 1st, 2nd, and 3rd rejection

- fraud-tagged approvals

- latency and processing behaviour across verification steps

**Qualitative**

- manual review of rejection cases

- pattern review of name mismatch examples

- image-quality failure patterns

- forged/edited document characteristics

- operational pain points for customer support/admin teams

**Key Findings**

- Rejection volume was high enough to materially impact activation and deposits.

- The single biggest rejection reason was name mismatch caused by formatting inconsistency, middle-name presence/absence, or surname ordering across Aadhaar and PAN.

- Drop-off risk increased meaningfully after repeated rejection attempts.

- Image-quality issues were a major OCR bottleneck.

- Gallery uploads introduced noise, spam and irrelevant image problems.

- Some approved documents were forged/edited, meaning the workflow needed stronger authenticity checks.

**Graphical Representation**

1) Rejection Breakdown


**Root Cause Framing**

The issue was not just “slow verification.” It was a combination of:

- rigid exact-match logic for names

- OCR confidence issues on poor-quality images

- noisy image submission behaviour

- weak document-authenticity checks

- insufficient control points around fraud and PII

- operational burden caused by repeated rejection loops

In short, the workflow was both too strict for genuine users and too weak against edited documents.

**Requirement Gathering & Vendor Alignment**

I translated the diagnostic findings into a structured requirement set and worked with Product and the vendor to evaluate feasibility across:

API & performance

- available verification APIs

- synchronous response design

- P95 / P96 / P98 response times

- cost per API call

- auth-token generation and request design

Data & logging requirements

- what metadata would be stored internally

- what logs should appear on the admin dashboard

- what signals operations teams needed for review/escalation

Risk & privacy controls

- vendor must not retain user details

- PII masking for internal visibility

- secure response handling

- auditability of rejection/approval outcomes

This turned the project into not just a UX improvement initiative, but a risk-aware systems redesign.

**Solution Design**

1) Fuzzy Name Matching

- Problem:
Exact string matching falsely rejected genuine users where Aadhaar and PAN contained middle-name differences, surname-order variations, or formatting inconsistencies.

- Change introduced:
A fuzzy matching rule was introduced so that if the normalized name-match score between Aadhaar and PAN was ≥ 66%, the case could pass this validation step, provided the rest of the document checks were valid.

- Why 66%?
This was treated as a decision threshold, not a confidence interval. The threshold was chosen through historical case review and threshold tuning to balance:

  - false rejection of genuine users

  - false approval of risky mismatches

2) OCR Improvement via Image Annotation

- Problem:
Low-confidence OCR output was driving a large volume of rejections.

- Change introduced:
Image annotation with bounding boxes was used to improve training data quality and extraction confidence.

- Expected effect:

   - reduce low-confidence image rejection

   - improve field extraction reliability

   - reduce failure caused by zoomed-out/blurry images

3) Disable Gallery Upload

- Problem:
Users were uploading spam, irrelevant, or low-quality images from gallery.

- Change introduced:
Gallery upload was disabled in favour of controlled live capture.

- Expected effect:

   - cleaner document input

   - lower invalid-image rates

   - better OCR consistency

4) Authenticated Document API + OTP Validation

- Problem:
Approximately 2,000 approved documents weekly were forged/edited.

- Change introduced:
An authenticated document API was integrated to:

- validate the document against trusted sources

- detect forged/edited images

- generate OTP to the phone number linked to Aadhaar

- reject non-authentic document images earlier in the flow

Expected effect:

- lower false negatives

- stronger authenticity control

- improved fraud detection

**Statistical Validation**
- What Was Validated -
 this was a fraud- and compliance-sensitive workflow, the project is best framed as a phased rollout / quasi-experimental validation, rather than a casual consumer-style A/B test.

The main performance questions were:

 - Did approval rate improve meaningfully?

 - Did latency reduce materially?

 - Did the system reduce friction without increasing fraud risk?

**Hypotheses**

For approval rate:

- H₀: Approval rate after redesign is the same as before redesign.

- H₁: Approval rate after redesign is higher than before redesign.

For verification time:

- H₀: Average verification time after redesign is the same as before redesign.

- H₁: Average verification time after redesign is lower than before redesign.

**Primary Metrics**

- approval rate

- verification time

- KYC-to-deposit conversion

- activated users after KYC completion

Guardrail Metrics

- forged-document pass rate

- false rejection rate

- support/admin review volume

- vendor API tail latency (P95/P98)

- PII/logging compliance

- escalation rate

**Confidence Intervals**

To show that the approval-rate improvement was not just random variation, we can estimate 95% confidence intervals around the pre- and post-change approval rates.

Baseline (before)
- documents uploaded = 55,000

- approved = 37,000

- approval rate = 67.3%

Approximate 95% CI:

p±1.96np(1−p)​
​

	​
95% CI ≈ 66.9% to 67.7%

Using
𝑝 = 0.6727
p=0.6727 and
𝑛
=
55
,
000
n=55,000:

95% CI ≈ 66.9% to 67.7%

After redesign

If weekly upload volume remained similar and approval rate rose to 81%:

Using
𝑝
=
0.81
p=0.81 and
𝑛
=
55
,
000
n=55,000:
n=55,000:

95% CI ≈ 80.7% to 81.3%

**Interpretation**

These intervals do not overlap, which strongly suggests that the uplift was statistically meaningful rather than due to sampling noise.

Approval uplift CI

 - Approximate uplift:

81.0% - 67.3% = 13.7 percentage points

 - Approximate 95% CI for uplift:

13.2 pp to 14.2 pp

This supports the conclusion that the redesigned workflow materially improved approval performance.

Note: These CI estimates assume comparable weekly traffic volume before and after rollout. In a production write-up, the exact post-launch sample size would be used.

**Threshold Selection Logic for 66%**

 - The 66% name-match rule should be described as a threshold optimization decision.

 | Similarity Threshold | Genuine Matches Approved | Risky Mismatches Approved | Operational Interpretation |
| -------------------- | ------------------------ | ------------------------- | -------------------------- |
| 50%                  | Very high                | Too high                  | Too lenient                |
| 60%                  | High                     | Moderate                  | Better, still risky        |
| 66%                  | High                     | Controlled                | Best balance               |
| 70%                  | Moderate                 | Low                       | Safer, more false rejects  |
| 80%                  | Low                      | Very low                  | Too strict                 |


Conceptual threshold evaluation as chosen after reviewing historical rejection patterns and testing how different cutoffs changed the balance between:

- genuine-user recovery

- fraud risk

- operational review burden

So the threshold was not “arbitrary”; it was selected as the best operating trade-off.

**Impact**

| Metric                      |  Before |   After |         Impact |
| --------------------------- | ------: | ------: | -------------: |
| Approval rate               |     68% |     81% |         +13 pp |
| Verification time           | ~25 sec | <10 sec |           -60% |
| Additional activated users  |       - |  +10.5K |  Growth uplift |
| Weekly deposit contribution |       - |  +$150K | Revenue uplift |


**Business Impact Summary**

- materially improved KYC pass-through for genuine users

- reduced onboarding friction

- accelerated deposits by bringing more users to activation

- strengthened fraud controls against forged/edited documents

- improved system speed and operational visibility

**Why This Project Matters**

This was not just a dashboarding or reporting exercise. It involved:

- problem diagnosis using user-level and funnel-level data

- fraud and false-negative analysis

- product requirement shaping

- vendor/API evaluation

- privacy and PII control thinking

- rule/threshold design

- operational logging requirements

- measurable business impact

It demonstrates a blend of:

- analytics

- product thinking

- risk awareness

- systems design

- program execution

**Key Takeaways**

- Rigid exact-match rules create avoidable friction
Identity systems must handle real-world formatting inconsistencies.

- OCR quality is a business lever, not just a technical detail
Better extraction quality directly influences approval rate and activation.

- Fraud control and UX must be designed together
Reducing rejection friction without strengthening authenticity checks would have been dangerous.

- Tail latency and operational logging matter
Fast APIs alone are not enough; support teams need the right metadata and visibility.

- Threshold decisions should be explained, not guessed
Rules like a 66% match threshold become credible when framed as tested operating cutoffs.




